In [1]:
import requests
import pandas as pd

# Fetch data from mfapi.in
url = "https://api.mfapi.in/mf/125497"
response = requests.get(url)
response.raise_for_status()  # raises an error if the request failed

data = response.json()

# Inspect the structure first
print(data['meta'])       # fund metadata (name, house, etc.)
print(data['data'][:5])   # first 5 NAV records

{'fund_house': 'SBI Mutual Fund', 'scheme_type': 'Open Ended Schemes', 'scheme_category': 'Equity Scheme - Small Cap Fund', 'scheme_code': 125497, 'scheme_name': 'SBI Small Cap Fund - Direct Plan - Growth', 'isin_growth': 'INF200K01T51', 'isin_div_reinvestment': None}
[{'date': '24-07-2026', 'nav': '204.85350'}, {'date': '23-07-2026', 'nav': '205.55390'}, {'date': '22-07-2026', 'nav': '207.67480'}, {'date': '21-07-2026', 'nav': '210.09000'}, {'date': '20-07-2026', 'nav': '209.44660'}]


In [2]:
# Convert the NAV history into a DataFrame
nav_df = pd.DataFrame(data['data'])

# Add fund identifying info from meta
nav_df['scheme_code'] = data['meta']['scheme_code']
nav_df['scheme_name'] = data['meta']['scheme_name']
nav_df['fund_house'] = data['meta']['fund_house']

print(nav_df.shape)
print(nav_df.head())
print(nav_df.dtypes)

(3129, 5)
         date        nav  scheme_code  \
0  24-07-2026  204.85350       125497   
1  23-07-2026  205.55390       125497   
2  22-07-2026  207.67480       125497   
3  21-07-2026  210.09000       125497   
4  20-07-2026  209.44660       125497   

                                 scheme_name       fund_house  
0  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
1  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
2  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
3  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
4  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
date             str
nav              str
scheme_code    int64
scheme_name      str
fund_house       str
dtype: object


In [3]:
output_path = '../data/raw/hdfc_top100_direct_nav_live.csv'
nav_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to ../data/raw/hdfc_top100_direct_nav_live.csv


In [1]:
import requests
import pandas as pd
import time

# Scheme codes to fetch
schemes = {
    "119551": "SBI Bluechip",
    "120503": "ICICI Bluechip",
    "118632": "Nippon Large Cap",
    "119092": "Axis Bluechip",
    "120841": "Kotak Bluechip",
}

all_dfs = []

for code, name in schemes.items():
    print(f"Fetching {name} ({code})...")
    url = f"https://api.mfapi.in/mf/{code}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data['data'])
    df['scheme_code'] = data['meta']['scheme_code']
    df['scheme_name'] = data['meta']['scheme_name']
    df['fund_house'] = data['meta']['fund_house']

    all_dfs.append(df)
    print(f"  -> {df.shape[0]} NAV records fetched")

    time.sleep(1)  # small delay to be polite to the API

print("\nDone fetching all 5 schemes.")

Fetching SBI Bluechip (119551)...
  -> 3274 NAV records fetched
Fetching ICICI Bluechip (120503)...
  -> 3345 NAV records fetched
Fetching Nippon Large Cap (118632)...
  -> 3336 NAV records fetched
Fetching Axis Bluechip (119092)...
  -> 3603 NAV records fetched
Fetching Kotak Bluechip (120841)...
  -> 3339 NAV records fetched

Done fetching all 5 schemes.


In [2]:
combined_df = pd.concat(all_dfs, ignore_index=True)

print(combined_df.shape)
print(combined_df['scheme_name'].value_counts())  # confirm all 5 funds present
print(combined_df.head())

(16897, 5)
scheme_name
HDFC Money Market Fund - Growth Option - Direct Plan                     3603
Axis ELSS Tax Saver Fund - Direct Plan - Growth Option                   3345
quant Mid Cap Fund - Growth Option - Direct Plan                         3339
Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option    3336
Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW           3274
Name: count, dtype: int64
         date        nav  scheme_code  \
0  24-07-2026  106.59610       119551   
1  23-07-2026  106.58510       119551   
2  22-07-2026  106.61070       119551   
3  21-07-2026  106.65670       119551   
4  20-07-2026  106.57370       119551   

                                         scheme_name  \
0  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
1  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
2  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
3  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
4  Aditya Birla Sun Life Banking &